## Policy evaluation — how AWS says yes or no

When a real request arrives, AWS runs every applicable policy through one evaluation, starting from an **implicit deny** — everything is denied until something allows it. The order:

1. **Explicit `Deny`** — checked across *every* policy. Any deny anywhere wins immediately, unconditionally. The kill switch.
2. **SCPs** — along the path Org root → OU → account. Must allow.
3. **Resource-based policies** — can grant on their own within the same account.
4. **Permission boundary** on the principal — must allow.
5. **Session policies** — must allow.
6. **Identity-based policies** — must allow.

**The single rule:** the effective permission is the **intersection** of every layer that has to say yes, and any explicit deny anywhere overrides all of it.

That deny-wins rule powers a key pattern — **permission boundaries for delegated administration**. Say you want a developer to create IAM roles for services they own, but *not* be able to create a role with `AdministratorAccess` and assume it. You grant them `iam:CreateRole` **with a condition** that every role they create must carry a specific permission boundary. The boundary grants nothing on its own — it only sets a ceiling — so whatever permissions policy they attach, the role can never exceed the cap. That's how you safely hand out IAM itself.